<a href="https://colab.research.google.com/github/python-freak/Coin-Classifier/blob/master/Coin_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA PREPROCESSING AND LOADING

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import os
import numpy as np
import cv2
from tqdm import tqdm

In [ ]:
base_dir = "/content/gdrive/My Drive/train"

In [ ]:
train_dataset = os.listdir("/content/gdrive/My Drive/train")

In [ ]:
coin_paths = [ os.path.join(base_dir,dir)  for dir in train_dataset]

In [ ]:
from collections import defaultdict

country2coins = defaultdict(list)

In [ ]:

for path in tqdm(coin_paths):
  country = (((path.split('/'))[-1]).split(','))[-1]
  for img in os.listdir(path):
    img_path = os.path.join(path,img)
    IMG = cv2.imread(img_path)
    IMG = cv2.resize(IMG, (128,128))
    IMG = np.array(IMG).astype('float32') / 255. 
    country2coins[country].append(IMG)

In [ ]:
country_labels = []
for key in country2coins:
  if key not in country_labels:
    country_labels.append(key)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
numeric_labels = le.fit_transform(country_labels)

label_dict = {country_labels[i]:numeric_labels[i] for i in range(len(country_labels))}


In [ ]:
reverse_label_dict = {numeric_labels[i]:country_labels[i] for i in range(len(country_labels))}

In [ ]:
X = []
Y_categorical = []
for key in country2coins:
  for img in country2coins[key]:
    X.append(img)
    Y_categorical.append(label_dict[key])

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(X[3000] , cmap='gray')

In [ ]:
# randominzing the data
import random
combined = list(zip(X, Y_categorical))
random.shuffle(combined)
X[:], Y_categorical[:] = zip(*combined)

In [ ]:
from keras.utils import to_categorical
Y = to_categorical(Y_categorical, num_classes = len(country_labels))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
X_train = np.array(X_train)
X_test  = np.array(X_test)

In [ ]:
print("X_train shape:{}".format(X_train.shape))
print("Y_train shape:{}".format(y_train.shape))
print("X_test shape:{}".format(X_test.shape))
print("Y_test shape:{}".format(y_test.shape))

# Model Building

In [ ]:
from keras.models import Model
from keras.layers import Input, Flatten, Dropout, BatchNormalization
from keras.layers import Dense, Conv2D, MaxPooling2D

In [ ]:
def classifier():
  z = Input(shape=(128,128,3))
  x = Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(128,128,3))(z)
  x = BatchNormalization()(x)

  x = Conv2D(32, kernel_size=(3, 3), activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPooling2D(pool_size = (2,2))(x)
  x = Dropout(0.2)(x)

  x = Conv2D(64, kernel_size=(3, 3), activation='relu')(x)
  x = BatchNormalization()(x)
  x = Dropout(0.2)(x)

  x = Conv2D(128, kernel_size=(3, 3), activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPooling2D(pool_size = (2,2))(x)
  x = Dropout(0.2)(x)

  x = Flatten()(x)
  x = Dense(512, activation = 'relu')(x)
  x = BatchNormalization()(x)
  x = Dropout(0.5)(x)

  x = Dense(128, activation = 'relu')(x)
  x = BatchNormalization()(x)
  x = Dropout(0.5)(x)

  output = Dense(32 , activation = 'softmax')(x)

  return Model(inputs = z , outputs = output )



In [ ]:
import tensorflow as tf
coin_classifier = classifier()
coin_classifier.compile(loss = 'categorical_crossentropy' , optimizer = 'adam' , metrics = ['accuracy'])

In [ ]:
coin_classifier.fit(X_train, y_train , epochs = 30 , validation_split = 0.1 , batch_size = 16 , shuffle = True)

In [ ]:
loss, accuracy = coin_classifier.evaluate(X_test, y_test)

In [ ]:
print("Test LOSS:{}".format(loss))
print("Test Accuracy:{}".format(accuracy*100))

In [ ]:
y_pred = coin_classifier.predict(X_test)

In [ ]:
y_pred_labels = []
y_test_labels = []
for y in y_pred:
  y_pred_labels.append(((np.where(y == y.max()))[0][0]))

for y in y_test:
  y_test_labels.append(((np.where(y == y.max()))[0][0]))

In [ ]:
predictions = []
for label in y_pred_labels:
  predictions.append(reverse_label_dict[label])



# TESTING 

In [ ]:
test_dir = "/content/gdrive/My Drive/test"

In [ ]:
def get_labels(predictions):
  _ = []
  for y in predictions:
    idx = ((np.where(y == y.max()))[0][0])
    _.append(reverse_label_dict[idx])

  return _

In [ ]:
test_results = {}
for img in tqdm(os.listdir(test_dir)):
  id = img.split('.')[0]
  img_path = os.path.join(test_dir,img)
  
  IMG = cv2.imread(img_path)
  IMG = cv2.resize(IMG,(128,128))
  IMG = np.array(IMG).astype('float32') / 255. 
  prediction = coin_classifier.predict(np.expand_dims(IMG, axis=0))
  label = (get_labels(prediction))[0]
  test_results[id] = label




In [ ]:
test_images_array = np.array(test_images)
test_predictions = coin_classifier.predict(test_images_array)
labels = get_labels(test_predictions)

In [ ]:
idx2labels = {i+1:labels[i] for i in range(len(labels))}

In [ ]:
import csv
with open('/content/gdrive/My Drive/output_1.csv','w') as f:
  for key in test_results.keys():
    f.write("%s,%s\n"%(key,test_results[key]))

In [ ]:
# saving the model
coin_classifier.save("/content/gdrive/My Drive/coin_classifier_85acc.h5")

In [ ]:
test_results